# 03 Test Siamese Network

This notebook tests the custom Siamese network on the payload loss test dataset. The model compares each current frame against an initial reference frame and outputs a payload loss score from 0 to 1, where higher scores indicate a higher likelihood of payload loss.

## 1. Imports and setup

In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.utils import GROUND_TRUTH_CSV, resolve_test_video_path
from src.siamese_method import SiameseMethod
from src.evaluate import evaluate_method

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Load Ground Truth

Load the test dataset annotations from `ground_truth.csv`.

In [4]:
gt = pd.read_csv(GROUND_TRUTH_CSV)
gt.head()

,filename,camera_id,is_loss_event,loss_frame,total_frames
0,videos_normal/indoor_full/front_empty_001.mp4,front,0,-1,410
1,videos_normal/indoor_full/back_empty_001.mp4,back,0,-1,408
2,videos_normal/indoor_full/left_empty_001.mp4,left,0,-1,406
3,videos_normal/indoor_full/right_empty_001.mp4,right,0,-1,404
4,videos_normal/indoor_med/front_empty_001.mp4,front,0,-1,411


## 3. Inspect One Video

Select one video from the dataset and run the Siamese network on it to check whether the score increases around the payload loss event.

In [5]:
# Pick one video to inspect
row = gt.iloc[0]
video_path = resolve_test_video_path(row["filename"])

row, video_path

(filename         videos_normal/indoor_full/front_empty_001.mp4
 camera_id                                                front
 is_loss_event                                                0
 loss_frame                                                  -1
 total_frames                                               410
 Name: 0, dtype: object,
 PosixPath('/Users/tohjiale/Desktop/payload_loss_detection/test_data/videos_normal/indoor_full/front_empty_001.mp4'))

## 4. Test Siamese network on one video

In [6]:
method = SiameseMethod(threshold=0.5, consecutive_frames=5)

result = method.predict_video(video_path)

print("Detected frame:", result["detected_frame"])
print("Ground truth loss frame:", row["loss_frame"])

Detected frame: -1
Ground truth loss frame: -1


## 5. Plot Siamese Score

The plot shows the Siamese network score over time. A higher score indicates a higher likelihood of payload loss.

In [ ]:
frames = list(range(1, len(result["scores"]) + 1))

plt.figure(figsize=(12, 4))
plt.plot(frames, result["scores"])
plt.axhline(method.threshold, linestyle="--", label="Threshold")

if row["is_loss_event"] == 1:
    plt.axvline(row["loss_frame"], linestyle="--", label="Ground Truth Loss Frame")

if result["detected_frame"] != -1:
    plt.axvline(result["detected_frame"], linestyle=":", label="Detected Frame")

plt.xlabel("Frame")
plt.ylabel("Siamese Loss Score")
plt.title(row["filename"])
plt.legend()
plt.savefig("results/fig13_siamese_detection.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Evaluate Siamese Network on Full Test Dataset

Run the Siamese network on all test videos and compute event-level, frame-level, and latency metrics.

In [ ]:
method = SiameseMethod(threshold=0.5, consecutive_frames=5)

video_results, frame_results, metrics = evaluate_method(method)

metrics

### Video-Level Results

Preview of evaluation results at the video level, including detected frame and ground truth labels.

In [ ]:
video_results.head()

### Frame-Level Results

Preview of per-frame predictions, scores, and inference latency for all videos.

In [ ]:
frame_results.head()

## 7. Threshold Sweep

The Siamese network outputs a score from 0 to 1, so this section tests different classification thresholds, from 0.10 to 0.90, in steps of 0.05.

In [ ]:
sweep_results = []

thresholds = [round(0.10 + i * 0.05, 2) for i in range(17)]

for threshold in thresholds:
    method = SiameseMethod(threshold=threshold, consecutive_frames=5)
    _, _, metrics = evaluate_method(method)

    sweep_results.append({
        "threshold": threshold,
        **metrics
    })

sweep_df = pd.DataFrame(sweep_results)
sweep_df

In [ ]:
# Plot threshold sweep

plt.figure(figsize=(10, 4))
plt.plot(sweep_df["threshold"], sweep_df["event_level_recall"], marker="o", label="Event-Level Recall")
plt.plot(sweep_df["threshold"], sweep_df["event_level_precision"], marker="o", label="Event-Level Precision")
plt.plot(sweep_df["threshold"], sweep_df["frame_level_precision"], marker="o", label="Frame-Level Precision")

plt.xlabel("Siamese Threshold")
plt.ylabel("Metric")
plt.title("Siamese Threshold Sweep")
plt.legend()
plt.grid(True)
plt.show()

## 8. Consecutive Frame Filtering

This tests how requiring multiple consecutive loss predictions affects false positives and detection reliability.

In [ ]:
filter_results = []

for consecutive_frames in [1, 3, 5, 10, 15, 30]:
    method = SiameseMethod(threshold=0.5, consecutive_frames=consecutive_frames)
    _, _, metrics = evaluate_method(method)

    filter_results.append({
        "consecutive_frames": consecutive_frames,
        **metrics
    })

filter_df = pd.DataFrame(filter_results)
filter_df

## 9. Save Results

Save Siamese network results for later comparison with SSIM and YOLO26n.

In [ ]:
output_dir = PROJECT_ROOT / "notebooks" / "results"
output_dir.mkdir(exist_ok=True)

video_results.to_csv(output_dir / "siamese_video_results.csv", index=False)
frame_results.to_csv(output_dir / "siamese_frame_results.csv", index=False)
sweep_df.to_csv(output_dir / "siamese_threshold_sweep.csv", index=False)
filter_df.to_csv(output_dir / "siamese_filter_sweep.csv", index=False)

print("Saved Siamese results.")

## 10. Conclusion

